# Lecture: Quantization — Smaller Numbers, Same Model

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain float vs integer formats in one sentence each: bits-per-knob sets the memory price (§4).
- Write the affine fake-quant round-trip on a worked example (§5): divide weights by a scale, round to integers, multiply back (`q = clamp(round(w/s))`, `w' = s·q`).
- Explain why small groups beat one global scale when outliers exist (§6).
- Read a memory-vs-quality frontier (val CE and weight error) and pick a format for a budget (§7–§8).


## The big idea

Model weights are long decimals like 0.5731, each normally costing 4 bytes (FP32). **Quantization** rounds them onto a short ruler with evenly spaced marks: pick a **scale** (the ruler's step size), divide, round to whole marks, multiply back when computing. Fewer marks fit in fewer bytes (INT8 = 1 byte, INT4 = half a byte), so the model shrinks 4–8x. The catch: one ruler for the whole shelf fails if a single giant book sets the scale — every small book rounds to zero. Fix: give each small shelf section (a **group** of 16–32 weights) its own ruler. Analogy: instead of measuring rice grains and watermelons with the same yardstick, use a teaspoon per bowl (groups) so nothing rounds to nothing.


In [ ]:
import numpy as np  # bring in numpy for the ruler positions
import matplotlib.pyplot as plt  # bring in pyplot for the ruler cartoon
np.random.seed(16)  # fix the seed so the toy weights match each run
clean = np.array([0.1, 0.3, 0.5, 0.7])  # four similar weights: one ruler fits all
out = np.array([0.1, 0.3, 0.5, 4.0])  # same shelf plus one watermelon outlier
s_clean = 0.7 / 7  # INT4-like scale from the local max (step 0.1)
s_out = 4.0 / 7  # one global scale stretched by the outlier (step 0.57)
print('clean scale:', round(float(s_clean), 3))  # print 0.1: every weight gets its own marks
print('outlier scale:', round(float(s_out), 3))  # print 0.571: small weights share ~1 mark
print('clean marks:', np.round(clean / s_clean).tolist())  # print [1, 3, 5, 7]: spread out
print('outlier marks:', np.round(out / s_out).tolist())  # print [0, 1, 1, 7]: small ones collapse
fig, ax = plt.subplots(figsize=(8, 3))  # wide flat figure for the two rulers
ax.scatter(clean, [1] * 4, s=120, label='clean shelf')  # top dots: well spread
ax.scatter(out, [0] * 4, s=120, label='shelf + outlier')  # bottom dots: three crushed at 0-1
ax.set_yticks([0, 1], ['with outlier', 'clean'])  # name the two shelves
ax.set_xlabel('weight value (ruler position)')  # label what the axis is
ax.set_title('One watermelon stretches the ruler (marks 1,3,5,7 -> 0,1,1,7)')  # title in printed numbers
ax.legend()  # key telling the shelves apart
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the cartoon
print('RECALIBRATE: constants above are the author CPU run (SEED=16); paste your results/quant.csv + quant_groups.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 prices the formats, §5–§6 round the weights, §7 reads the damage.

| Term | One-liner | Section |
|---|---|---|
| FP32 / BF16 / FP16 | Decimal formats at 4 / 2 / 2 bytes per weight; fewer bits = smaller but coarser | §4 |
| INT8 / INT4 | Whole-number formats at 1 / 0.5 bytes per weight after rounding with a scale | §4 |
| Scale | The ruler step: `max / top_mark`; outliers stretch it for everyone | §5 |
| Fake-quant | Simulate rounding in software (round-trip through integers) without fast kernels | §5 |
| Group size | How many weights share one ruler (g16 = a ruler per 16 weights) | §6 |
| Val CE / weight error | Quality after rounding (noisy here) / mean damage per weight (honest signal) | §7 |


### §4 Formats — bits per knob set the price

**In one sentence: FP32 stores each weight in 4 bytes, BF16/FP16 in 2, INT8 in 1, INT4 in half a byte — so the same 112,500-knob proxy costs 0.45 MB down to 0.056 MB, an 8x shrink.**

Floats (FP32/BF16/FP16) keep decimals with different range/precision trade-offs: BF16 keeps FP32's huge range with a short mantissa (big numbers safe, fine detail coarse), FP16 has a tighter range (watch overflow). Integers (INT8/INT4) keep no decimals at all — just whole marks plus one shared scale. The price list is pure multiplication: bytes-per-knob × knob count. Watch it: the bars falling 0.45 → 0.056 MB.

> FP8-E4M3 (in the lab's 7-format `quant.csv`, not the 4 bars above): an 8-bit float at 1 byte/knob needing per-tensor scaling — the CPU fake is a cast round-trip (`torch.float8_e4m3fn`; if your torch build lacks it, drop the FP8 row, the INT ladder is the lesson). NVFP4 (§6) = FP4 values + FP8 scales at group-16.

In [ ]:
import numpy as np  # reuse numpy for the format arrays
import matplotlib.pyplot as plt  # reuse pyplot for the bits-per-knob bars
np.random.seed(16)  # fix the seed for reproducibility
fmts = ['FP32', 'BF16', 'INT8', 'INT4']  # four ruler styles from fine to coarse
bpp = np.array([4.0, 2.0, 1.0, 0.5])  # bytes charged per weight per format
mb = np.array([0.450, 0.225, 0.113, 0.056])  # MB for the 112,500-knob proxy
print('bytes/param:', bpp.tolist())  # show the price list halving each step
print('MB:', mb.tolist())  # show memory shrinking 8x by INT4
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the bars
ax.bar(fmts, mb, color='tab:blue')  # draw memory collapsing 8x
ax.set_ylabel('MB (lower is smaller)')  # label defines the memory direction
ax.set_xlabel('format (finer -> coarser)')  # label names the ordering
ax.set_title('Memory shrinks 8x by INT4 (0.45 -> 0.056 MB)')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §5 Scale and fake-quant — snap to the nearest mark

**In one sentence: affine fake-quant picks `scale = max / top_mark`, snaps each weight with `q = clamp(round(w / scale))`, and steps back with `w' = scale · q` — the rounded decimals the model actually uses.**

Walk the numbers: weights [−0.9, −0.35, 0.1, 0.42, 0.95] with an INT4-like top mark 7 give scale 0.95/7 ≈ 0.136, marks [−7, −3, 1, 3, 7], and rounded decimals [−0.95, −0.407, 0.136, 0.407, 0.95] — close marks mean tiny errors. **Fake**-quant means we simulate this round-trip in software (compute still runs in float); real speed needs hardware kernels that compute in the narrow type directly. Watch it: the number line plus every mark printed.


In [ ]:
import numpy as np  # reuse numpy for the rounding math
import matplotlib.pyplot as plt  # reuse pyplot for the number line
np.random.seed(16)  # fix the seed so the toy weights match each run
w = np.array([-0.9, -0.35, 0.1, 0.42, 0.95])  # five toy weights on a line
s = 0.95 / 7  # INT4-like scale: max weight divided by top mark 7
q = np.round(w / s)  # snap each weight to its nearest ruler mark
wq = q * s  # step back to decimals: the rounded weights actually used
print('true:', w.tolist())  # show original decimals before rounding
print('scale:', round(float(s), 4))  # print 0.1357: one step of the ruler
print('marks:', q.tolist())  # show whole-number marks each weight snapped to
print('rounded:', [round(float(x), 3) for x in wq])  # show decimals after rounding
fig, ax = plt.subplots(figsize=(8, 2.5))  # make a wide flat strip like a ruler
ax.scatter(w, [1] * len(w), s=120, label='true')  # top dots are original weights
ax.scatter(wq, [0] * len(wq), s=120, label='rounded')  # bottom dots are snapped weights
ax.set_yticks([0, 1], ['rounded', 'true'])  # name the two rows in plain words
ax.set_xlabel('weight value (ruler position)')  # label what the axis is
ax.set_title('Quantization = snap to the nearest mark')  # title in plain words
ax.legend()  # key telling true dots from rounded dots
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the number line

### §6 Groups — small rulers beat one global ruler

**In one sentence: the group size is how many weights share one scale, and shrinking it from per-tensor (all weights, one ruler) to g16 (a ruler per 16 weights) cuts INT4 damage from 0.0141 to 0.0087 (~40%) because each ruler adapts to its local sizes.**

Recall the anchor cartoon: one global max stretches every mark. Per-group scales fix exactly that — an outlier stretches only its own group of 16, while the other groups keep fine steps for their small weights. The ladder is monotonic: per-tensor 0.0141 → g128 0.0114 → g32 0.0098 → g16 0.0087 (mean damage per weight). NVFP4-style microscaling is this same idea pushed further (tiny blocks, each with its own small scale). Watch it: the ladder bars with every rung printed.


In [ ]:
import numpy as np  # reuse numpy for the ladder arrays
import matplotlib.pyplot as plt  # reuse pyplot for the ladder bars
np.random.seed(16)  # fix the seed for reproducibility
groups = ['per-tensor', 'g128', 'g32', 'g16']  # ruler sharing from global to tiny
gerr = [0.0141, 0.0114, 0.0098, 0.0087]  # measured INT4 damage per granularity
print('group errors:', gerr)  # show smaller groups damage weights less
print('g16 saves:', round(100 * (0.0141 - 0.0087) / 0.0141, 0), '% vs per-tensor')  # print 38.0%: the grouping win
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the ladder
ax.bar(groups, gerr, color='tab:red')  # draw damage falling as rulers shrink
ax.set_ylabel('mean weight error (lower is better)')  # label defines the damage
ax.set_xlabel('weights sharing one ruler')  # label defines group size simply
ax.set_title('Smaller groups -> less damage (0.0141 -> 0.0087)')  # title in printed numbers
ax.tick_params(axis='x', rotation=15)  # tilt labels so they fit
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the ladder

### §7 Val CE and weight error — reading the frontier honestly

**In one sentence: val CE (validation loss after rounding) barely moves here because 60 training steps leave ±0.02 batch noise, so the mean weight error — average damage per weight — is the honest signal, and it ladders cleanly: BF16 ~0, INT8 small, INT4 needs groups.**

The frontier plots memory (x) against damage (y): down-left is better. BF16 halves memory for ~free (damage 0.0002); INT8 quarters it (0.0008); INT4 eighths it but damage jumps to 0.0098 at g32 — rescued to 0.0087 at g16 (§6). Declaring “INT4 is free” from flat val CE would be lying with noise; the werr ladder is the claim you can defend. Watch it: the frontier scatter plus the damage line, all printed.


In [ ]:
import numpy as np  # reuse numpy for the frontier arrays
import matplotlib.pyplot as plt  # reuse pyplot for scatter + line
np.random.seed(16)  # fix the seed for reproducibility
fmts = ['FP32', 'BF16', 'INT8', 'INT4-g32']  # four stops on the frontier
mem = np.array([0.450, 0.225, 0.113, 0.056])  # weight MB per format
werr = np.array([0.0, 0.0002, 0.0008, 0.0098])  # mean damage per weight per format
dce = np.array([0.0, 0.002, 0.008, 0.031])  # val-CE drift vs FP32 (inside noise: read, do not trust)
print('werr:', werr.tolist())  # show the honest ladder: 0.0 -> 0.0098
print('dCE:', dce.tolist())  # show val drift stays small here (brief training)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: frontier + damage line
ax[0].scatter(mem, werr)  # frontier scatter: down-left is better
ax[0].set_xlabel('weight memory (MB)')  # label the price axis
ax[0].set_ylabel('mean weight error')  # label the damage axis
ax[0].set_title('Frontier: INT4 cheapest, most damage')  # title the trade-off
ax[0].grid(True, alpha=0.3)  # add faint grid to read values
for f, m, e in zip(fmts, mem, werr):  # label every frontier point
    ax[0].annotate(f, (m, e), fontsize=8)  # format name at its point
ax[1].plot(fmts, werr, marker='o', color='tab:red')  # damage line across formats
ax[1].set_ylabel('mean weight error (lower is better)')  # label defines damage
ax[1].set_title('Damage ladders: 0.0 -> 0.0098')  # title in printed numbers
ax[1].tick_params(axis='x', rotation=15)  # tilt labels so they fit
plt.tight_layout()  # stop titles and labels from overlapping
plt.show()  # render the figure

## §8 How it all fits together — the system map

One format trial is a pipeline with every § term playing a station. **Weights** (FP32, §4) pick a **scale** from their max (§5), **round** to integer marks (INT8 one scale, INT4 one scale per **group**, §6), **dequantize** back to decimals for compute, and the harness reads **val CE + weight error** (§7) to place the format on the frontier. The map below pins each term to its station; the trace walks 4 weights through INT4 by hand.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(16)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('weights\nFP32 §4', 1), ('scale\nmax/7 §5', 3.8), ('round\ngroups §6', 6.6), ('dequant\ns·q §5', 9.4), ('damage\nCE+werr §7', 12)]  # five pipeline stations
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = forward pipeline
for xa, xb in [(1.9, 2.9), (4.7, 5.7), (7.5, 8.5), (10.3, 11.1)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2),  # draw a forward arrow left -> right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.text(6.5, 3.4, 'FORWARD: pick ruler, snap, step back (§4-§6)', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(6.5, 0.8, 'SCORE: werr ladders honestly, val CE sits in noise (§7)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # scoring banner
ax.set_title('System map: weights -> scale -> round -> dequant -> damage')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the INT4 round-trip
np.random.seed(16)  # fix the seed for reproducibility
w = np.array([0.62, -0.31, 0.05, -0.88])  # four weights sharing one INT4 group
s = 0.88 / 7  # group scale: local max over top mark 7
q = np.round(w / s)  # snap to whole marks
wq = q * s  # step back to decimals
print('scale:', round(float(s), 4))  # print 0.1257: this group's ruler step
print('marks:', q.tolist())  # print [5, -2, 0, -7]: nearest marks
print('rounded:', [round(float(x), 3) for x in wq])  # print [0.629, -0.251, 0.0, -0.88]
print('errors:', [round(float(x), 3) for x in np.abs(wq - w)])  # print [0.009, 0.059, 0.05, 0.0]: small survives

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the pipeline above, on a real briefly-trained TinyGPT:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 16` habit as this lecture.
2. **Cell 2 (data)** — TinyShakespeare (or synthetic fallback), 90/10 split — the weights our ruler rounds.
3. **Cell 3 (fake-quant library)** — codes `q = clamp(round(w/s))`, `w' = s·q` for INT8/INT4 (§5) plus float cast round-trips (§4), with unit tests asserting the error ordering (§7's ladder).
4. **Cell 4 (7-format table)** — re-scores the same weights in FP32/BF16/FP16/FP8/INT8/INT4/NVFP4 into `results/quant.csv`: MB (§4) + val CE + werr (§7).
5. **Cell 5 (group sweep + figure)** — retries per-tensor → g128 → g32 → g16 (§6: 0.0141 → 0.0087) into `results/quant_groups.csv` and `results/quant.png`.
6. **Cell 6 (cleanup)** — warns the latency column is CPU fake-quant (dequant runs in fp32): memory/quality transfer, kernel speed needs a GPU rerun.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **INT4 uses 0.5 bytes/weight vs FP32's 4. Why is the model ~8x smaller?**

*Answer: each weight's storage shrinks 8x (§4: 0.45 → 0.056 MB); the tiny extra rulers (one scale per group, §6) add almost nothing. Trace the §8 arithmetic: 4 weights, one shared scale, zero extra marks.*

2. **Why do small groups (g16) beat one global scale (0.0087 vs 0.0141)?**

*Answer: one global scale is stretched by the single biggest weight (§5's watermelon), so small weights all round to ~zero; per-group rulers (§6) adapt to local sizes and keep small weights alive — a 38% damage cut.*

3. **Val loss barely moves across formats here — can we declare INT4 free?**

*Answer: no (§7). At 60 training steps all val-CE deltas sit inside ±0.02 batch noise, so the weight-error ladder — not val CE — is the honest signal; a fully trained model would show real quality gaps.*
